# Task 15: Distributed Data Parallel (DDP) multi-GPU/node training

Needs multiple GPUs (or multiple containers) to actually launch. Writes the training script + docker-compose file and shows the launch commands.

In [1]:
%%writefile train_ddp.py
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

def setup():
    dist.init_process_group(backend="nccl")  # use "gloo" for CPU-only nodes
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    return local_rank

def cleanup():
    dist.destroy_process_group()

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x):
        return self.net(x)

def main():
    local_rank = setup()
    device = torch.device(f"cuda:{local_rank}")

    model = Net().to(device)
    model = DDP(model, device_ids=[local_rank])
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    X = torch.randn(512, 20, device=device)
    y = torch.randn(512, 1, device=device)

    for epoch in range(10):
        opt.zero_grad()
        loss = nn.functional.mse_loss(model(X), y)
        loss.backward()
        opt.step()
        if local_rank == 0:
            print(epoch, loss.item())

    cleanup()

if __name__ == "__main__":
    main()


Writing train_ddp.py


In [2]:
%%writefile docker-compose.yml
version: "3.8"
services:
  node0:
    image: pytorch/pytorch:latest
    command: torchrun --nnodes=2 --nproc_per_node=1 --node_rank=0 --master_addr=node0 --master_port=29500 train_ddp.py
    volumes:
      - ./:/workspace
    working_dir: /workspace
    deploy:
      resources:
        reservations:
          devices:
            - capabilities: [gpu]
  node1:
    image: pytorch/pytorch:latest
    command: torchrun --nnodes=2 --nproc_per_node=1 --node_rank=1 --master_addr=node0 --master_port=29500 train_ddp.py
    volumes:
      - ./:/workspace
    working_dir: /workspace
    deploy:
      resources:
        reservations:
          devices:
            - capabilities: [gpu]


Writing docker-compose.yml


In [3]:
# launch (single node, 2 GPUs):
# torchrun --standalone --nproc_per_node=2 train_ddp.py

# launch (multi-node cluster defined above):
# docker-compose up
